# 01 — Environment & Intelligence
## Agentic X | Studio 67

This notebook prepares the simulated event environment, validates the scenario dataset, defines agent and environment characteristics, and executes the Creative LLM Agent.

The system handles event crises through four collaborating agents: Creative, Finance, Operations, and Safety.


Cell 2 — Connect to GitHub

In [1]:

from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/ProjectsSpaceHub/AgenticX.git"
ROOT = Path("/content/AgenticX")

if not ROOT.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(ROOT)],
        check=True
    )

os.chdir(ROOT)

print("Project directory:", Path.cwd())


Project directory: /content/AgenticX


Cell 3 — Install Requirements

In [2]:

import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt"
    ],
    check=True
)

print("Dependencies installed successfully")


Dependencies installed successfully


Cell 4 — Import Libraries

In [3]:

import pandas as pd
from IPython.display import display

from core.environment import (
    DEFAULT_DATA,
    ENVIRONMENT_DESCRIPTION,
    HIDDEN_OPTION_FIELDS,
    HIDDEN_CASE_FIELDS,
    load_dataset,
    get_case
)

from agents.creative import CreativeAgent

print("All imports completed")


All imports completed


Cell 5 — Load the Dataset


In [4]:

cases_df, options_df = load_dataset()

print("Total scenarios:", len(cases_df))
print("Total solution options:", len(options_df))
print("Unique cases:", cases_df["case_id"].nunique())

display(cases_df.head())
display(options_df.head())


Total scenarios: 150
Total solution options: 450
Unique cases: 150


,case_id,scenario,incident,budget_sar,time_left_min,severity,backup_available,crew_available,venue_access,notes,...,initial_budget_sar,committed_spend_sar,incident_type,required_quality_min,safety_policy,case_profile,crew_count_available,unplanned_charge_sar,projected_total_spend_sar,projected_overrun_sar
0,SCR-001,Screen Failure,Main LED screen failed during final rehearsal,8500,100,Critical,True,True,True,Synthetic test input; no precomputed agent dec...,...,8500,0,display_failure,60,Reject non-compliant options; escalate if none...,Baseline feasible,3,0,0,0
1,SCR-002,Screen Failure,LED wall lost signal before keynote,2400,60,High,False,True,True,Synthetic test input; no precomputed agent dec...,...,2400,0,display_failure,50,Reject non-compliant options; escalate if none...,Tight budget feasible,2,0,0,0
2,SCR-003,Screen Failure,Display controller failure before exhibition o...,7220,90,High,True,True,True,Synthetic test input; no precomputed agent dec...,...,7220,0,display_failure,50,Reject non-compliant options; escalate if none...,Time pressure feasible,2,0,0,0
3,SCR-004,Screen Failure,Screen power module stopped working,14000,150,Medium,False,True,True,Synthetic test input; no precomputed agent dec...,...,14000,0,display_failure,50,Reject non-compliant options; escalate if none...,Resource-constrained feasible,3,0,0,0
4,SCR-005,Screen Failure,Main stage screen shows intermittent blackout,11000,150,Critical,True,True,True,Synthetic test input; no precomputed agent dec...,...,11000,0,display_failure,55,Reject non-compliant options; escalate if none...,Multi-option feasible,1,0,0,0


,case_id,option_id,option_name,cost_sar,duration_min,safety_ok,quality_score,needs_backup,needs_crew,requires_venue_access,...,crew_required,risk_score_after_mitigation,safety_rejection_reason,safety_assessment_basis,gross_savings_sar,net_savings_sar,projected_total_after_sar,projected_overrun_after_sar,addresses_incident,resolution_explanation
0,SCR-001,SCR-001-O1,Use backup LED,1290,40,True,89,True,True,True,...,1,1,None — safety conditions satisfied,"Check safety_ok, permit_ok, mitigation_verifie...",0,NaN,NaN,NaN,True,Addresses the display failure via alternative ...
1,SCR-001,SCR-001-O2,Rent replacement screen,5920,70,True,89,False,True,True,...,1,1,None — safety conditions satisfied,"Check safety_ok, permit_ok, mitigation_verifie...",0,NaN,NaN,NaN,True,Addresses the display failure via alternative ...
2,SCR-001,SCR-001-O3,Continue with projection,1210,25,True,60,False,True,True,...,1,3,None — safety conditions satisfied,"Check safety_ok, permit_ok, mitigation_verifie...",0,NaN,NaN,NaN,True,Addresses the display failure via alternative ...
3,SCR-002,SCR-002-O1,Use backup LED,1230,40,True,75,True,True,True,...,1,2,None — safety conditions satisfied,"Check safety_ok, permit_ok, mitigation_verifie...",0,NaN,NaN,NaN,True,Addresses the display failure via alternative ...
4,SCR-002,SCR-002-O2,Rent replacement screen,6300,80,True,96,False,True,True,...,1,3,None — safety conditions satisfied,"Check safety_ok, permit_ok, mitigation_verifie...",0,NaN,NaN,NaN,True,Addresses the display failure via alternative ...


Cell 6 — Validate the Dataset

In [5]:

assert len(cases_df) == 150
assert len(options_df) == 450

assert cases_df["case_id"].is_unique
assert options_df["option_id"].is_unique

assert not cases_df["case_id"].isna().any()
assert not options_df["case_id"].isna().any()

assert set(options_df["case_id"]).issubset(
    set(cases_df["case_id"])
)

options_per_case = options_df.groupby("case_id").size()

assert (options_per_case == 3).all()

print("Dataset validation passed")
print("\nScenario distribution:")
print(cases_df["scenario"].value_counts())


Dataset validation passed

Scenario distribution:
scenario
Screen Failure     50
Budget Overrun     50
Safety Conflict    50
Name: count, dtype: int64


Cell 7 — Define the Simulated Environment.

In [6]:

environment = {
    "name": "Studio 67",
    "domain": "Event Crisis Management",
    "agents": [
        "Creative",
        "Finance",
        "Operations",
        "Safety"
    ],
    "resources": [
        "Budget",
        "Time",
        "Crew",
        "Equipment",
        "Venue"
    ],
    "scenario_types": sorted(
        cases_df["scenario"].unique().tolist()
    )
}

for key, value in environment.items():
    print(f"{key}: {value}")


name: Studio 67
domain: Event Crisis Management
agents: ['Creative', 'Finance', 'Operations', 'Safety']
resources: ['Budget', 'Time', 'Crew', 'Equipment', 'Venue']
scenario_types: ['Budget Overrun', 'Safety Conflict', 'Screen Failure']


Cell 8 — Environment Classification

In [26]:
classification = {
    "Observability": (
        "Partially Observable",
        "Agents receive only the available incident and resource information."
    ),
    "Determinism": (
        "Deterministic Simulation",
        "The current dataset uses fixed case conditions; "
        "LLM assessments may vary between runs."
    ),
    "Dynamics": (
        "Static per Evaluation",
        "The case state remains fixed during each evaluation. "
        "Future simulation steps may update time and resources."
    ),
    "State Space": (
        "Discrete",
        "Agents select from a finite set of solution options. "
        "Time and cost are numeric attributes."
    )
}

classification_df = pd.DataFrame(
    [
        {
            "Property": key,
            "Classification": value[0],
            "Justification": value[1]
        }
        for key, value in classification.items()
    ]
)

display(classification_df)

,Property,Classification,Justification
0,Observability,Partially Observable,Agents receive only the available incident and...
1,Determinism,Deterministic Simulation,The current dataset uses fixed case conditions...
2,Dynamics,Static per Evaluation,The case state remains fixed during each evalu...
3,State Space,Discrete,Agents select from a finite set of solution op...


Cell 9 — Creative Agent PEAS

In [27]:
creative_peas = {
    "Performance Measure": (
        "Incident relevance, solution quality, "
        "and audience experience"
    ),
    "Environment": (
        "Studio 67 event crisis simulation"
    ),
    "Actuators": (
        "Evaluate solutions and return structured "
        "approval, score, and reasoning"
    ),
    "Sensors": (
        "Incident description, available solutions, "
        "and quality requirements"
    )
}

peas_df = pd.DataFrame(
    creative_peas.items(),
    columns=["PEAS Component", "Description"]
)

display(peas_df)

print("Agent Type: Deliberative")
print(
    "Justification: Creative uses LLM reasoning to "
    "evaluate alternatives before providing its assessment."
)

,PEAS Component,Description
0,Performance Measure,"Incident relevance, solution quality, and audi..."
1,Environment,Studio 67 event crisis simulation
2,Actuators,Evaluate solutions and return structured appro...
3,Sensors,"Incident description, available solutions, and..."


Agent Type: Deliberative
Justification: Creative uses LLM reasoning to evaluate alternatives before providing its assessment.


Cell 10 — Select a Test Scenario

In [9]:

CASE_ID = "SCR-001"

case, options = get_case(CASE_ID)

print("Case ID:", case["case_id"])
print("Scenario:", case["scenario"])
print("Incident:", case["incident"])
print("Available budget:", case["initial_budget_sar"])
print("Time remaining:", case["time_left_min"])

print("\nAvailable solutions:")

display(
    pd.DataFrame(options)[
        [
            "option_id",
            "option_name",
            "cost_sar",
            "duration_min",
            "quality_score"
        ]
    ]
)


Case ID: SCR-001
Scenario: Screen Failure
Incident: Main LED screen failed during final rehearsal
Available budget: 8500
Time remaining: 100

Available solutions:


,option_id,option_name,cost_sar,duration_min,quality_score
0,SCR-001-O1,Use backup LED,1290,40,89
1,SCR-001-O2,Rent replacement screen,5920,70,89
2,SCR-001-O3,Continue with projection,1210,25,60


Cell 11 — Prevent Evaluation Data Leakage

In [10]:

assert "case_profile" not in case

for option in options:
    assert "addresses_incident" not in option

    for field in HIDDEN_OPTION_FIELDS:
        assert field not in option

print("No evaluation ground-truth labels exposed")
print("Creative will assess the options independently")


No evaluation ground-truth labels exposed
Creative will assess the options independently


Cell 12 — Configure LLM Access

In [14]:
%pip install -q -U langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.4.0 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.


In [19]:
import os
from getpass import getpass

os.environ.pop("OPENAI_API_KEY", None)

os.environ["GOOGLE_API_KEY"] = getpass(
    "Enter your Gemini API key: "
)

os.environ["AGENTICX_MODEL"] = "gemini-3.8-flash"
print("Gemini API configured successfully")

Enter your Gemini API key: ··········
Gemini API configured successfully


In [20]:

from pathlib import Path

creative_code = '''
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from core.models import AgentEvaluation


class CreativeOpinion(BaseModel):
    approved: bool
    score: float = Field(ge=0, le=100)
    reason: str


class CreativeAgent:
    name = "Creative"

    def __init__(self):
        if not os.getenv("GOOGLE_API_KEY"):
            raise RuntimeError("GOOGLE_API_KEY is missing")

        self.llm = ChatGoogleGenerativeAI(
            model=os.getenv("AGENTICX_MODEL", "gemini-2.5-flash"),
            temperature=0
        ).with_structured_output(
            CreativeOpinion,
            method="json_schema"
        )

    def evaluate(self, case, option):
        prompt = (
            "You are the Creative Agent for Studio 67. "
            "Evaluate whether the solution addresses the incident "
            "and maintains event quality and audience experience. "
            "Reject irrelevant solutions and do not invent facts.\\n"
            f"CASE: {case}\\n"
            f"OPTION: {option}"
        )

        answer = self.llm.invoke(prompt)

        quality_ok = (
            float(option.get("quality_score") or 0)
            >= float(case.get("required_quality_min") or 0)
        )

        return AgentEvaluation(
            agent=self.name,
            option_id=str(option["option_id"]),
            approved=bool(answer.approved and quality_ok),
            score=float(answer.score),
            reason=(
                answer.reason if quality_ok
                else "Below minimum quality requirement"
            ),
            blocking=False
        )
'''

Path("agents/creative.py").write_text(
    creative_code,
    encoding="utf-8"
)

print("Creative Agent updated to Gemini")


Creative Agent updated to Gemini


Cell 13 — Initialize Creative Agent

In [21]:
import importlib
import agents.creative as creative_module

importlib.reload(creative_module)

creative_agent = creative_module.CreativeAgent()

print("Creative Agent initialized with Gemini")

Creative Agent initialized with Gemini


Cell 14 — Evaluate the Three Solutions

In [22]:

creative_results = []

for option in options:

    print("Evaluating:", option["option_name"])

    evaluation = creative_agent.evaluate(
        case,
        option
    )

    creative_results.append(
        evaluation.model_dump()
    )

results_df = pd.DataFrame(creative_results)

display(results_df)


Evaluating: Use backup LED
Evaluating: Rent replacement screen
Evaluating: Continue with projection


,agent,option_id,approved,score,reason,blocking
0,Creative,SCR-001-O1,True,89.0,The solution effectively replaces the failed m...,False
1,Creative,SCR-001-O2,True,89.0,Renting a replacement screen directly resolves...,False
2,Creative,SCR-001-O3,True,60.0,The projection option successfully addresses t...,False


Cell 15 — Validate Creative Outputs

In [23]:

assert len(results_df) == len(options)

assert results_df["option_id"].is_unique

assert results_df["score"].between(0, 100).all()

assert results_df["reason"].notna().all()

assert (
    results_df["agent"] == "Creative"
).all()

print("Creative output validation passed")


Creative output validation passed


Cell 16 — Display Creative Recommendations


In [24]:

ranked_results = results_df.sort_values(
    by="score",
    ascending=False
)

display(
    ranked_results[
        [
            "option_id",
            "approved",
            "score",
            "reason"
        ]
    ]
)

approved_results = ranked_results[
    ranked_results["approved"]
]

if approved_results.empty:
    print("Creative did not approve any solution.")
else:
    best = approved_results.iloc[0]

    print("Creative preferred option:", best["option_id"])
    print("Creative score:", best["score"])

print(
    "\nNote: Final selection requires Finance, "
    "Operations and Safety approval."
)


,option_id,approved,score,reason
0,SCR-001-O1,True,89.0,The solution effectively replaces the failed m...
1,SCR-001-O2,True,89.0,Renting a replacement screen directly resolves...
2,SCR-001-O3,True,60.0,The projection option successfully addresses t...


Creative preferred option: SCR-001-O1
Creative score: 89.0

Note: Final selection requires Finance, Operations and Safety approval.


Cell 17 — Save Results

In [25]:

from pathlib import Path

output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / "creative_evaluations.csv"

results_df.to_csv(
    output_path,
    index=False
)

print("Results saved to:", output_path)


Results saved to: results/creative_evaluations.csv


In [28]:
additional_results = []

scenario_types = [
    "Budget Overrun",
    "Safety Conflict"
]

for scenario_type in scenario_types:

    matching = cases_df[
        cases_df["scenario"] == scenario_type
    ]

    assert not matching.empty, f"No cases for {scenario_type}"

    test_case_id = matching.iloc[0]["case_id"]
    test_case, test_options = get_case(test_case_id)

    # Evaluate one real option from each additional scenario.
    option = test_options[0]

    evaluation = creative_agent.evaluate(
        test_case,
        option
    )

    additional_results.append({
        "scenario": scenario_type,
        **evaluation.model_dump()
    })

additional_df = pd.DataFrame(additional_results)

display(additional_df)

assert len(additional_df) == 2
assert additional_df["score"].between(0, 100).all()
assert additional_df["reason"].str.strip().ne("").all()

print("Additional scenario evaluations completed")

,scenario,agent,option_id,approved,score,reason,blocking
0,Budget Overrun,Creative,BDG-001-O1,True,64.0,Reducing decorative lighting preserves the cor...,False
1,Safety Conflict,Creative,SFT-001-O1,False,0.0,The proposed option completely fails to addres...,False


Additional scenario evaluations completed


## Conclusion

This notebook prepared and validated the Studio 67 event crisis dataset, described the simulated environment, and documented the Creative Agent using PEAS.

The Creative Agent uses a real LLM to evaluate alternative solutions based on incident relevance and event quality.

Its evaluations are structured and validated before being passed to the multi-agent coordination process.

Final decisions require collaboration with Finance, Operations, and Safety.